In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [97]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
root = "c:\\Architectural-Biases-in-Time-Series-Anomaly-Detection"
def create_dict(name):
    path = os.path.join(root, "saved_model_scores", name)
    with np.load(path) as data:
       results = {name: metric for name, metric in data.items()}
    return results
lstm_f_results = create_dict("lstm_f_results.npz")
lstm_ae_results = create_dict("lstm_ae_results.npz")
transformer_f_results = create_dict("transformer_f_results.npz")

In [122]:
f_scores = lstm_f_results["v_scores"]
f_labels = lstm_f_results["v_labels"]
f_cats = lstm_f_results["v_categories"]

transformer_scores = transformer_f_results["v_scores"]
transformer_labels = transformer_f_results["v_labels"]
transformer_cats = transformer_f_results["v_categories"]

ae_scores = lstm_ae_results["v_scores"]
ae_labels = lstm_ae_results["v_labels"]
ae_cats = lstm_ae_results["v_categories"]

window_grid = np.linspace(1, 800, 10)

In [ ]:
def generate_df(scores, labels, cats, window_grid):
    series = []

    for window in window_grid:
        window = int(window)
        current_score = np.convolve(scores, np.ones(window) / window, mode='same')
        threshold = np.percentile(current_score, 100 - 4.4)
        cat_dict = {}
        normal_mask = cats == 0
        FPR = np.sum(current_score[normal_mask] > threshold) / np.sum(normal_mask)
        cat_dict["FPR (0.0)"] = round(float(FPR), 2)

        for cat in np.unique(cats):
            if cat == 0.0:
                continue
            cat_mask = cats == cat
            recall = np.sum(current_score[cat_mask] > threshold) / np.sum(cat_mask)
            cat_dict[f"recall ({cat})"] = round(float(recall), 2)

        TP = np.sum((current_score > threshold) & (labels == 1))
        FP = np.sum((current_score > threshold) & (labels == 0))
        recall = TP / np.sum(labels == 1)
        precision = TP / (TP + FP)
        F1 = 2 * (precision * recall) / (precision + recall)
        cat_dict["F1"] = np.round(F1, 3)

        series.append(pd.Series(cat_dict, name=f"t={threshold:.2f}|w={window}"))

    return pd.concat(series, axis=1)
generate_df(f_scores, f_labels, f_cats, window_grid)


,t=33.55|w=1,t=29.59|w=89,t=28.42|w=178,t=27.48|w=267,t=27.52|w=356,t=27.99|w=444,t=29.47|w=533,t=30.96|w=622,t=33.91|w=711,t=39.97|w=800
FPR (0.0),0.030,0.020,0.020,0.020,0.02,0.020,0.020,0.020,0.020,0.020
recall (1.0),0.750,0.860,0.880,0.900,0.92,0.940,0.960,0.970,0.980,0.990
recall (2.0),0.190,0.280,0.310,0.330,0.33,0.340,0.350,0.320,0.340,0.350
recall (3.0),0.660,0.840,0.940,0.940,0.97,0.990,1.000,0.930,0.910,0.910
recall (4.0),0.420,0.500,0.550,0.590,0.62,0.650,0.670,0.700,0.690,0.700
recall (5.0),0.040,0.020,0.010,0.010,0.02,0.030,0.040,0.040,0.050,0.060
recall (6.0),0.140,0.240,0.260,0.300,0.33,0.370,0.380,0.360,0.290,0.270
recall (7.0),0.390,0.460,0.480,0.510,0.53,0.550,0.550,0.540,0.510,0.480
recall (8.0),0.680,0.790,0.830,0.840,0.84,0.850,0.860,0.870,0.870,0.850
recall (9.0),0.400,0.710,0.840,0.920,0.93,0.930,0.910,0.930,0.870,0.790


In [125]:
generate_df(ae_scores, ae_labels, ae_cats, window_grid)

,t=8524.06|w=1,t=8411.13|w=89,t=8222.49|w=178,t=8040.41|w=267,t=7839.24|w=356,t=7756.42|w=444,t=7694.68|w=533,t=7606.03|w=622,t=7654.15|w=711,t=7739.72|w=800
FPR (0.0),0.020,0.020,0.02,0.020,0.020,0.020,0.02,0.020,0.020,0.020
recall (1.0),0.900,0.910,0.92,0.940,0.950,0.970,0.98,0.980,0.990,0.990
recall (2.0),0.200,0.200,0.20,0.210,0.230,0.250,0.25,0.230,0.210,0.180
recall (3.0),0.710,0.740,0.79,0.810,0.830,0.840,0.85,0.860,0.810,0.810
recall (4.0),0.470,0.470,0.50,0.530,0.550,0.560,0.56,0.570,0.580,0.590
recall (5.0),0.110,0.110,0.12,0.140,0.150,0.150,0.16,0.180,0.190,0.180
recall (6.0),0.300,0.310,0.32,0.360,0.380,0.390,0.38,0.380,0.380,0.340
recall (7.0),0.440,0.460,0.48,0.510,0.520,0.510,0.51,0.510,0.510,0.510
recall (8.0),0.740,0.750,0.76,0.770,0.770,0.780,0.80,0.800,0.810,0.800
recall (9.0),0.480,0.520,0.57,0.620,0.680,0.710,0.71,0.720,0.700,0.670


In [126]:
generate_df(transformer_scores, transformer_labels, transformer_cats, window_grid)


,t=32.96|w=1,t=27.81|w=89,t=27.60|w=178,t=29.39|w=267,t=31.74|w=356,t=35.25|w=444,t=39.43|w=533,t=49.68|w=622,t=72.59|w=711,t=92.82|w=800
FPR (0.0),0.020,0.020,0.020,0.010,0.010,0.010,0.020,0.020,0.020,0.020
recall (1.0),0.780,0.870,0.890,0.900,0.920,0.950,0.960,0.970,0.980,0.990
recall (2.0),0.270,0.320,0.360,0.340,0.330,0.350,0.290,0.300,0.300,0.310
recall (3.0),0.800,0.920,0.940,0.960,0.910,0.910,0.910,0.910,0.910,0.790
recall (4.0),0.460,0.530,0.560,0.580,0.610,0.640,0.640,0.660,0.680,0.690
recall (5.0),0.200,0.300,0.320,0.360,0.360,0.380,0.360,0.310,0.300,0.260
recall (6.0),0.160,0.250,0.300,0.320,0.290,0.270,0.260,0.210,0.220,0.230
recall (7.0),0.540,0.700,0.750,0.750,0.760,0.740,0.680,0.610,0.470,0.400
recall (8.0),0.760,0.850,0.870,0.880,0.880,0.890,0.880,0.850,0.820,0.820
recall (9.0),0.540,0.820,0.900,0.910,0.920,0.860,0.780,0.770,0.780,0.660
